# Comparing two houses, before and after renovation

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [106]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [107]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.26) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [108]:
renderer = "vscode"

##4.01 Importin the OBJ file

In [115]:
Ground_floor_outer_boundary = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\outer boundary_ground floor.dxf")
Ground_floor_outer_boundary = Wire.ByEdges(Ground_floor_outer_boundary)
print(Wire.IsClosed(Ground_floor_outer_boundary))

print(Wire.IsClosed(Ground_floor_outer_boundary), Wire.IsManifold(Ground_floor_outer_boundary))

Ground_floor_outer_boundary_closed = Wire.Close(Ground_floor_outer_boundary)
print(Wire.IsClosed(Ground_floor_outer_boundary_closed))


print(Wire.IsClosed(Ground_floor_outer_boundary_closed), Wire.IsManifold(Ground_floor_outer_boundary_closed))

ground_floor_inner_boundary = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\inner boundary_ground floor.dxf")
print(ground_floor_inner_boundary)
cluster = Cluster.ByTopologies(ground_floor_inner_boundary)
ground_floor_inner_boundary = Topology.SelfMerge(cluster)


ground_floor_inner_boundary = Topology.Wires(ground_floor_inner_boundary)   

Ground_floor = Face.ByWire(Ground_floor_outer_boundary, ground_floor_inner_boundary)



False
False False
Wire.Close - Error: The input wire parameter contains less than two open end vertices. Returning None.
caller name: <module>
None
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: <module>
None None
[<topologic_core.Edge object at 0x00000230B4B55F30>, <topologic_core.Edge object at 0x00000230B4B57A30>, <topologic_core.Edge object at 0x00000230B4B54DF0>, <topologic_core.Edge object at 0x00000230B4B55FB0>, <topologic_core.Edge object at 0x00000230B4B57AF0>, <topologic_core.Edge object at 0x00000230B4B55B30>, <topologic_core.Edge object at 0x00000230B4B56DB0>, <topologic_core.Edge object at 0x00000230B64AD1B0>, <topologic_core.Edge object at 0x00000230B64AE4F0>, <topologic_core.Edge object at 0x00000230B64AF470>, <topologic_core.Edge object at 0x00000230B64AE9F0>, <topologic_core.Edge object at 0x00000230B64AD330>, <topologic_core.Edge object at 0x00000230B64AF3B0>, <topologic_core.Edge object at 0x00000230B64AE

a lot of claude code to fix the open wire, it turns out there on only one open vertex of the wire

In [116]:
outer_edges = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\outer boundary_ground floor.dxf")
print(f"Number of edges from DXF: {len(outer_edges)}")

# Build the wire
w = Wire.ByEdges(outer_edges)
print(f"IsClosed: {Wire.IsClosed(w)}")
print(f"Number of edges in wire: {len(Topology.Edges(w))}")
print(f"Number of vertices in wire: {len(Topology.Vertices(w))}")

Number of edges from DXF: 323
IsClosed: False
Number of edges in wire: 323
Number of vertices in wire: 323


In [117]:
outer_edges = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\outer boundary_ground floor.dxf")
print(f"Edges from DXF: {len(outer_edges)}")

# Use SelfMerge to properly stitch and split into separate wires
cluster = Cluster.ByTopologies(outer_edges)
merged = Topology.SelfMerge(cluster)
wires = Topology.Wires(merged)

print(f"Number of separate wires after SelfMerge: {len(wires)}")
for i, w in enumerate(wires):
    edges_in_wire = len(Topology.Edges(w))
    print(f"  Wire {i}: closed={Wire.IsClosed(w)}, edges={edges_in_wire}")

Edges from DXF: 323
Topology.Wires - Warning: The input is a Wire. Returning the same wire embedded in a list.
caller name: <module>
Number of separate wires after SelfMerge: 1
  Wire 0: closed=False, edges=323


In [118]:


# Get all vertices in the wire and count edge connections
all_vertices = Topology.Vertices(w)
print(f"Total vertices: {len(all_vertices)}")

# Find open endpoints (vertices touching only 1 edge)
open_endpoints = []
for v in all_vertices:
    incident_edges = Topology.SuperTopologies(v, w, topologyType="edge")
    if len(incident_edges) == 1:
        open_endpoints.append(v)
        print(f"  Open endpoint at: ({Vertex.X(v):.6f}, {Vertex.Y(v):.6f}, {Vertex.Z(v):.6f})")

print(f"Found {len(open_endpoints)} open endpoints")

Total vertices: 323
  Open endpoint at: (30116.325195, -1058.196782, 8530.516379)
Found 1 open endpoints


In [119]:
from collections import Counter


all_vertices = Topology.Vertices(w)

# Count incident edges for each vertex
counts = []
for v in all_vertices:
    n = len(Topology.SuperTopologies(v, w, topologyType="edge"))
    counts.append(n)

distribution = Counter(counts)
print(f"Edges-per-vertex distribution: {dict(distribution)}")

# Show any vertex that's not exactly 2-connected
for v in all_vertices:
    n = len(Topology.SuperTopologies(v, w, topologyType="edge"))
    if n != 2:
        print(f"  Vertex with {n} edges at: ({Vertex.X(v):.4f}, {Vertex.Y(v):.4f}, {Vertex.Z(v):.4f})")

Edges-per-vertex distribution: {2: 321, 3: 1, 1: 1}
  Vertex with 3 edges at: (30116.3252, -1028.1968, 8530.5164)
  Vertex with 1 edges at: (30116.3252, -1058.1968, 8530.5164)


In [120]:


open_v = open_endpoints[0]
all_edges = Topology.Edges(w)

# Find the edge touching the open vertex
stray_edge = None
for e in all_edges:
    for ev in Topology.Vertices(e):
        if Vertex.Distance(ev, open_v) < 0.001:
            stray_edge = e
            break
    if stray_edge:
        break

# Rebuild without it
clean_edges = [e for e in all_edges if e != stray_edge]
print(f"Edges before: {len(all_edges)}, after: {len(clean_edges)}")

cluster = Cluster.ByTopologies(clean_edges)
merged = Topology.SelfMerge(cluster)
fixed_wires = Topology.Wires(merged)
Ground_floor_outer_boundary = fixed_wires[0]
print("Closed now?", Wire.IsClosed(Ground_floor_outer_boundary))

Edges before: 323, after: 322
Topology.Wires - Warning: The input is a Wire. Returning the same wire embedded in a list.
caller name: <module>
Closed now? True


building the ground floor face

In [121]:
Ground_floor = Face.ByWire(Ground_floor_outer_boundary, ground_floor_inner_boundary)

## 6. Show the geometry

In [ ]:
Topology.Show(Ground_floor,
              faceColor=[210,210,250],
              faceOpacity=0.5,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer = renderer)

creating the grid overlay

In [146]:
b_r = Wire.BoundingRectangle(Ground_floor)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+100,100))
vRange = list(range(0,int(length)+5000,100))

grid = Grid.EdgesByDistances(Ground_floor, clip=True, uRange=uRange, vRange=vRange)

print("width:", width)
print("length:", length)
print("uRange:", uRange)
print("vRange:", vRange)

width: 6647.280153
length: 3244.586894
uRange: [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400, 2500, 2600, 2700, 2800, 2900, 3000, 3100, 3200, 3300, 3400, 3500, 3600, 3700, 3800, 3900, 4000, 4100, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 4900, 5000, 5100, 5200, 5300, 5400, 5500, 5600, 5700, 5800, 5900, 6000, 6100, 6200, 6300, 6400, 6500, 6600, 6700]
vRange: [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400, 2500, 2600, 2700, 2800, 2900, 3000, 3100, 3200, 3300, 3400, 3500, 3600, 3700, 3800, 3900, 4000, 4100, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 4900, 5000, 5100, 5200, 5300, 5400, 5500, 5600, 5700, 5800, 5900, 6000, 6100, 6200, 6300, 6400, 6500, 6600, 6700, 6800, 6900, 7000, 7100, 7200, 7300, 7400, 7500, 7600, 7700, 7800, 7900, 8000, 8100, 8200]


showing the geometry and the grid

In [147]:
Topology.Show(Ground_floor, grid,
              camera=[0,0,20],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Derive primal graph

In [6]:
cluster_old = Cluster.ByTopologies(cc_old)
vertices_old = Topology.Vertices(cluster_old)
print(vertices_old)
vertices_old = [Topology.Copy(v) for v in vertices_old]
edges_old = Topology.Edges(cluster_old)
edges_old = [Topology.Copy(e) for e in edges_old]
primal_graph_old = Graph.ByVerticesEdges(vertices_old , edges_old)

Cluster.ByTopologies - Warning: The input topologies parameter contains only one topology. Returning the same topology.
caller name: <module>
[<topologic_core.Vertex object at 0x000001AB14B603B0>, <topologic_core.Vertex object at 0x000001AB129832B0>, <topologic_core.Vertex object at 0x000001AB143ABB70>, <topologic_core.Vertex object at 0x000001AB143ABD30>, <topologic_core.Vertex object at 0x000001AB143ABDB0>, <topologic_core.Vertex object at 0x000001AB143ABE30>, <topologic_core.Vertex object at 0x000001AB143ABE70>, <topologic_core.Vertex object at 0x000001AB143ABEB0>, <topologic_core.Vertex object at 0x000001AB143ABEF0>, <topologic_core.Vertex object at 0x000001AB143ABDF0>, <topologic_core.Vertex object at 0x000001AB143ABF30>, <topologic_core.Vertex object at 0x000001AB143ABF70>, <topologic_core.Vertex object at 0x000001AB143ABFB0>, <topologic_core.Vertex object at 0x000001AB14CF0030>, <topologic_core.Vertex object at 0x000001AB14CF0070>, <topologic_core.Vertex object at 0x000001AB14CF

## 8. Assign visual attributes to the vertices and edges of the graph

In [7]:
vertices = Graph.Vertices(primal_graph_old)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(primal_graph_old)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    e = Topology.SetDictionary(e, d)

## 9. Show the geometry and the graph

In [8]:
Topology.Show(cc_old, primal_graph_old,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 10. Derive dual graph (based on cells in the cellcomplex)

In [9]:
dual_graph_old = Graph.ByTopology(cc_old, direct=True)

## 11. Assign visual attributes to the vertices and edges of the graph

In [10]:
vertices = Graph.Vertices(dual_graph_old)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(dual_graph_old)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "red"])
    e = Topology.SetDictionary(e, d)

## 12. Show the geometry and graph

In [11]:
Topology.Show(cc_old,dual_graph_old,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

Import doors as OBJ file

In [12]:
doors_old = Topology.ByOBJPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\01\murgle old house\murgle_old_house_doors.obj")
doors_old=Cluster.Faces(Cluster.ByTopologies(doors_old))

# zosto ne raboti slednata linija
#Cluster.Faces(door_old)





Adding the doors as apertures

In [13]:
cc_old = Topology.AddApertures(cc_old, doors_old, subTopologyType="face")

Deriving the dual graph witht he doors added

In [14]:
dual_graph_old = Graph.ByTopology(cc_old,direct=False, viaSharedApertures=True)

Assigning the visual stuff

In [15]:
vertices = Graph.Vertices(dual_graph_old)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(dual_graph_old )
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "red"])
    e = Topology.SetDictionary(e, d)

Show the geometry plus the graph

In [16]:
Topology.Show(cc_old, doors_old,dual_graph_old,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

Importing windows as obj file

In [17]:
windows_old = Topology.ByOBJPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\01\murgle old house\murgle_old_house_windows.obj")

windows_old=Cluster.Faces(Cluster.ByTopologies(windows_old))

adding the doors and the windows as apertures

In [27]:
cc_old = Topology.AddApertures(cc_old, doors_old+windows_old, subTopologyType="face")

Deriving the dual graph with the doors and the windows added

In [32]:
dual_graph_old = Graph.ByTopology(cc_old,direct=False, viaSharedApertures=True,toExteriorApertures=True)


adding visuals

In [50]:
vertices = Graph.Vertices(dual_graph_old)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges_old = Graph.Edges(dual_graph_old )
total_length = 0
midpoint_vertices = []
for e in edges_old:
    length = round(Edge.Length(e), 1) 
    print(f"Edge length: {length}")
    total_length += length 
    d = Dictionary.ByKeysValues(["width", "color","length"], [4, "red",length])
    e = Topology.SetDictionary(e, d)
      # Create midpoint vertex at 0.5 parameter along the edge
    mid_v = Edge.VertexByParameter(e, 0.5)
    d_mid = Dictionary.ByKeysValues(["size", "color", "label"], [1, "black", str(length)])
    mid_v = Topology.SetDictionary(mid_v, d_mid)
    midpoint_vertices.append(mid_v)
print(f"Total edge length: {total_length}")


Edge length: 183.8
Edge length: 228.6
Edge length: 143.7
Edge length: 207.9
Edge length: 213.7
Edge length: 192.3
Edge length: 183.8
Edge length: 64.9
Edge length: 190.9
Edge length: 125.3
Edge length: 70.1
Edge length: 148.9
Edge length: 159.5
Edge length: 179.0
Edge length: 291.8
Edge length: 211.5
Edge length: 185.8
Edge length: 186.8
Edge length: 232.3
Edge length: 183.8
Edge length: 189.8
Edge length: 234.1
Edge length: 228.8
Edge length: 127.9
Total edge length: 4365.0


In [40]:
# After the loop, rebuild the graph with updated edges
dual_graph_old = Graph.ByVerticesEdges(vertices, edges_old)

showing the geometry plus the graph with the doors and the windows

In [65]:
Topology.Show(cc_old, doors_old,windows_old,dual_graph_old, edges_old,mid_v, midpoint_vertices,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              edgeLabelKey="length",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer,
              showVertices=True,
             vertexLabelKey="label",
              showVertexLabel=True,
             
             
             )

               

Deriving the dual graph with doors 

In [66]:
dual_graph_old = Graph.ByTopology(cc_old,direct=False, viaSharedApertures=True)

Adding visuals

In [67]:
vertices = Graph.Vertices(dual_graph_old)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges_old = Graph.Edges(dual_graph_old )
total_length = 0
midpoint_vertices = []
for e in edges_old:
    length = round(Edge.Length(e), 1) 
    print(f"Edge length: {length}")
    total_length += length 
    d = Dictionary.ByKeysValues(["width", "color","length"], [4, "red",length])
    e = Topology.SetDictionary(e, d)
      # Create midpoint vertex at 0.5 parameter along the edge
    mid_v = Edge.VertexByParameter(e, 0.5)
    d_mid = Dictionary.ByKeysValues(["size", "color", "label"], [1, "black", str(length)])
    mid_v = Topology.SetDictionary(mid_v, d_mid)
    midpoint_vertices.append(mid_v)
print(f"Total edge length: {total_length}")

Edge length: 232.3
Edge length: 186.8
Edge length: 127.9
Edge length: 179.0
Edge length: 228.8
Edge length: 159.5
Edge length: 234.1
Edge length: 70.1
Edge length: 291.8
Edge length: 148.9
Edge length: 211.5
Edge length: 143.7
Edge length: 189.8
Edge length: 228.6
Total edge length: 2632.7999999999997


Showing the graph

In [69]:
Topology.Show(cc_old, doors_old,dual_graph_old, edges_old,mid_v, midpoint_vertices,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              edgeLabelKey="length",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer,
              showVertices=True,
             vertexLabelKey="label",
              showVertexLabel=True,
             
             
             )

Import the obj file from the new house

In [70]:
objects_new = Topology.ByOBJPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\01\murgle new house\murgle_new_house_walls.obj", selfMerge=True)
cells_new = Topology.Cells(objects_new[0])

cc_new = CellComplex.ByCells(cells_new)

the dual graph creating for the new house

In [71]:
dual_graph_new = Graph.ByTopology(cc_new, direct=True)

In [ ]:
importing the doors from the new house

In [72]:
doors_new = Topology.ByOBJPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\01\murgle new house\murgle_new_house_doors.obj")
doors_new=Cluster.Faces(Cluster.ByTopologies(doors_new))

In [ ]:
adding the doors as apertures to the cc_new

In [73]:
cc_new = Topology.AddApertures(cc_new, doors_new, subTopologyType="face")

importin the windows of the new house

In [74]:
windows_new = Topology.ByOBJPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\01\murgle new house\murgle_new_house_windows.obj")

windows_new=Cluster.Faces(Cluster.ByTopologies(windows_new))

adding the doors and the windows as apertures to the new house

In [86]:
cc_new = Topology.AddApertures(cc_new, doors_new+windows_new, subTopologyType="face")

In [ ]:
deriving the new dual graph with the appertures

In [87]:

dual_graph_new = Graph.ByTopology(cc_new,direct=False, viaSharedApertures=True,toExteriorApertures=True)

adding visuals for the new house

In [91]:
vertices = Graph.Vertices(dual_graph_new)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "green"])
    v = Topology.SetDictionary(v, d)
edges_new = Graph.Edges(dual_graph_new )
total_length = 0
midpoint_vertices = []
for e in edges_new:
    length = round(Edge.Length(e), 1) 
    print(f"Edge length: {length}")
    total_length += length 
    d = Dictionary.ByKeysValues(["width", "color","length"], [4, "green",length])
    e = Topology.SetDictionary(e, d)
      # Create midpoint vertex at 0.5 parameter along the edge
    mid_v = Edge.VertexByParameter(e, 0.5)
    d_mid = Dictionary.ByKeysValues(["size", "color", "label"], [1, "black", str(length)])
    mid_v = Topology.SetDictionary(mid_v, d_mid)
    midpoint_vertices.append(mid_v)
print(f"Total edge length: {total_length}")

Edge length: 190.0
Edge length: 163.5
Edge length: 138.4
Edge length: 131.2
Edge length: 507.3
Edge length: 387.6
Edge length: 376.0
Edge length: 395.0
Edge length: 391.9
Edge length: 566.1
Edge length: 620.8
Edge length: 452.4
Edge length: 213.7
Edge length: 183.7
Edge length: 228.8
Edge length: 206.3
Edge length: 186.6
Total edge length: 5339.3


Showing the  new house and the new graph

In [92]:
Topology.Show(cc_new, doors_new,windows_new,dual_graph_new, edges_new,mid_v_new, midpoint_vertices,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              edgeLabelKey="length",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer,
              
             vertexLabelKey="label",
              showVertexLabel=True,
             
             
             )

New house access graph + doors + endge lenght 

In [93]:
dual_graph_new = Graph.ByTopology(cc_new,direct=False, viaSharedApertures=True)

In [94]:
vertices = Graph.Vertices(dual_graph_new)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "green"])
    v = Topology.SetDictionary(v, d)
edges_new = Graph.Edges(dual_graph_new )
total_length = 0
midpoint_vertices = []
for e in edges_new:
    length = round(Edge.Length(e), 1) 
    print(f"Edge length: {length}")
    total_length += length 
    d = Dictionary.ByKeysValues(["width", "color","length"], [4, "green",length])
    e = Topology.SetDictionary(e, d)
      # Create midpoint vertex at 0.5 parameter along the edge
    mid_v = Edge.VertexByParameter(e, 0.5)
    d_mid = Dictionary.ByKeysValues(["size", "color", "label"], [1, "black", str(length)])
    mid_v = Topology.SetDictionary(mid_v, d_mid)
    midpoint_vertices.append(mid_v)
print(f"Total edge length: {total_length}")

Edge length: 620.8
Edge length: 163.5
Edge length: 452.4
Edge length: 566.1
Edge length: 507.3
Edge length: 228.8
Edge length: 206.3
Edge length: 138.4
Total edge length: 2883.6000000000004


In [95]:
Topology.Show(cc_new, doors_new,windows_new,dual_graph_new, edges_new,mid_v_new, midpoint_vertices,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              edgeLabelKey="length",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer,
              
             vertexLabelKey="label",
              showVertexLabel=True,
             
             
             )

In [ ]:
New house access graph via doors

In [98]:
dual_graph_new = Graph.ByTopology(cc_new,direct=False, viaSharedApertures=True)

In [99]:
vertices = Graph.Vertices(dual_graph_new)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "green"])
    v = Topology.SetDictionary(v, d)
edges_new = Graph.Edges(dual_graph_new )

for e in edges_new:
     
    d = Dictionary.ByKeysValues(["width", "color"], [4, "green"])
    e = Topology.SetDictionary(e, d)
      

In [100]:
Topology.Show(cc_new, doors_new, dual_graph_new,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

New house access graph 

In [101]:
dual_graph_new = Graph.ByTopology(cc_new,direct=True)

In [102]:
vertices = Graph.Vertices(dual_graph_new)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "green"])
    v = Topology.SetDictionary(v, d)
edges_new = Graph.Edges(dual_graph_new )

for e in edges_new:
     
    d = Dictionary.ByKeysValues(["width", "color"], [4, "green"])
    e = Topology.SetDictionary(e, d)

In [103]:
Topology.Show(cc_new, doors_new, dual_graph_new,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

overlapping the new house and the old house with the graphs

In [111]:
# Color old house edges
for edge in CellComplex.Edges(cc_old):
    Topology.SetDictionary(edge, Dictionary.ByKeysValues(["color"], ["red"]))

# Color new house edges
for edge in CellComplex.Edges(cc_new):
    Topology.SetDictionary(edge, Dictionary.ByKeysValues(["color"], ["green"]))

In [114]:
Topology.Show(
    doors_new, dual_graph_new,
    doors_old, dual_graph_old,
    cc_old, cc_new,
    vertexSizeKey="size",
    vertexColorKey="color",
    edgeWidthKey="width",
    edgeColorKey="color",
    edgeWidth=2, 
    opacityKey="FakeKey",
    faceOpacityKey="FakeKey",
    faceOpacity=0.3,
    backgroundColor="white",
    width=500,
    height=500,
    renderer=renderer
)